In [ ]:
# ======================================================
# Notebook: Drug Combination Optimisation (Logistic Regression)
# Maximise (- side effects)
# ======================================================

import numpy as np
from sklearn.linear_model import LogisticRegression

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (N,3)
y_raw = np.load("/mnt/data/initial_outputs.npy") # (N,)

# Transform objective (lower side effects = better)
y_transformed = -y_raw

# Convert to binary target (top 25% safest combinations)
threshold = np.percentile(y_transformed, 75)
y_bin = (y_transformed >= threshold).astype(int)

# Train logistic regression surrogate
model = LogisticRegression(max_iter=1000)
model.fit(X, y_bin)

# Candidate grid within observed bounds
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(3)]
grid = [np.linspace(b[0], b[1], 20) for b in bounds]
X_grid = np.array(np.meshgrid(*grid)).T.reshape(-1,3)

# Predict probability of low side effects
probs = model.predict_proba(X_grid)[:,1]

# Select next (10,3) combinations
top_idx = np.argsort(probs)[-10:]
next_points = X_grid[top_idx]

print("Next (10,3) compound combinations:")
print(next_points)